# 02. Test Retrieval Strategies

Compare semantic vs hybrid retrieval methods.


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import time
from config.settings import EmbeddingConfig, RetrieverConfig
from src.embeddings import MiniLMEmbedder
from src.retrieval import HybridRetriever
from sklearn.metrics.pairwise import cosine_similarity

/Users/soham/Desktop/nutribot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data


In [2]:
df = pd.read_csv('../data/processed/nutrition_docs.csv')
documents = df['content'].tolist()
print(f"Loaded {len(documents)} documents")

Loaded 10 documents


## Initialize Hybrid Retriever


In [3]:
embedder = MiniLMEmbedder(device='cpu')
retriever_config = RetrieverConfig(
    top_k=5,
    bm25_weight=0.3,
    semantic_weight=0.7,
    use_hybrid_search=True
)
print("Initializing hybrid retriever...")
retriever = HybridRetriever(documents, embedder, retriever_config)
print("Hybrid retriever ready!")

2026-06-15 20:18:53,700 - src.embeddings.minilm - INFO - Loading MiniLM embedder from sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 24553.19it/s]
2026-06-15 20:18:56,429 - src.embeddings.minilm - INFO - MiniLM embedder loaded successfully on cpu
2026-06-15 20:18:56,431 - src.retrieval.hybrid - INFO - Initializing hybrid retriever with 10 documents
2026-06-15 20:18:56,432 - src.retrieval.hybrid - INFO - BM25 index initialized


Initializing hybrid retriever...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.96it/s]
2026-06-15 20:18:56,489 - src.retrieval.hybrid - INFO - Embedded 10 documents in 0.06s


Hybrid retriever ready!


## Test Retrieval


In [4]:
test_queries = [
    "What are the benefits of protein?",
    "How much water should I drink?",
    "weight loss tips",
]

for query in test_queries:
    print(f"\\nQuery: {query}")
    print("=" * 60)
    start = time.time()
    results = retriever.retrieve(query, top_k=3)
    elapsed = time.time() - start
    for i, result in enumerate(results):
        print(f"\\n{i+1}. (score: {result.score:.3f}, {elapsed*1000:.1f}ms)")
        print(f"   {result.content[:100]}...")
        print(f"   BM25: {result.metadata['bm25_score']:.3f}, Semantic: {result.metadata['semantic_score']:.3f}")

2026-06-15 20:19:03,760 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:19:03,761 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:19:03,768 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:19:03,769 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:19:03,775 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:19:03,775 - src.retrieval.hybrid - INFO - Retrieved 2 relevant documents


\nQuery: What are the benefits of protein?
\n1. (score: 0.677, 15.3ms)
   Protein is essential for muscle growth and repair. The recommended daily intake is 0.8g per kilogram...
   BM25: 0.983, Semantic: 0.546
\n2. (score: 0.534, 15.3ms)
   Healthy fats from sources like avocados, nuts, and olive oil are important for hormone production an...
   BM25: 1.000, Semantic: 0.334
\n3. (score: 0.463, 15.3ms)
   Carbohydrates are the body's primary energy source. Complex carbs like whole grains provide sustaine...
   BM25: 0.983, Semantic: 0.240
\nQuery: How much water should I drink?
\n1. (score: 0.795, 7.6ms)
   Water intake recommendations vary, but the general guideline is 8 glasses per day. However, needs de...
   BM25: 1.000, Semantic: 0.707
\n2. (score: 0.342, 7.6ms)
   Sugar consumption should be minimized. Added sugars provide empty calories and can lead to weight ga...
   BM25: 0.400, Semantic: 0.317
\n3. (score: 0.337, 7.6ms)
   Sodium intake should be limited to less than 2,300mg p

## Analyze Score Distribution


In [5]:
all_results = []
for query in test_queries:
    results = retriever.retrieve(query, top_k=5)
    for result in results:
        all_results.append({
            'query': query,
            'combined_score': result.score,
            'bm25_score': result.metadata['bm25_score'],
            'semantic_score': result.metadata['semantic_score'],
        })
scores_df = pd.DataFrame(all_results)
print("Score Distribution:")
print(scores_df.describe())

2026-06-15 20:19:07,403 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:19:07,404 - src.retrieval.hybrid - INFO - Retrieved 4 relevant documents
2026-06-15 20:19:07,411 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.01s
2026-06-15 20:19:07,411 - src.retrieval.hybrid - INFO - Retrieved 3 relevant documents
2026-06-15 20:19:07,417 - src.retrieval.hybrid - INFO - Hybrid retrieval completed in 0.00s
2026-06-15 20:19:07,417 - src.retrieval.hybrid - INFO - Retrieved 2 relevant documents


Score Distribution:
       combined_score  bm25_score  semantic_score
count        9.000000    9.000000        9.000000
mean         0.504291    0.704696        0.418403
std          0.177039    0.371357        0.151570
min          0.302776    0.000000        0.240000
25%          0.342060    0.448720        0.317177
50%          0.462969    0.983231        0.359870
75%          0.677110    1.000000        0.539947
max          0.794875    1.000000        0.706964
